# Jury LLM: Multi-Model & Human-in-the-Loop Evaluation System

This notebook serves as the interactive frontend for the Jury System.

In [3]:
import sys
import os
sys.path.append('../')

from dotenv import load_dotenv
load_dotenv('../.env')

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from src.graph import app
from src.utils import parse_json_output
from src.llm_provider import LLMProvider
from src.agents import QUALIFICATION_PROMPT
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

llm = LLMProvider()


INFO:src.llm_provider:Using DashScope configuration.
INFO:src.llm_provider:Using DashScope configuration.


## Step 1: Input Target Text
Please paste the LLM generated text you want to evaluate below. This will be stored globally for the entire session.

In [6]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr

# --- Global Variables ---
TARGET_TEXT = ""
HUMAN_COMPETENCY_SCORE = 0
QUALIFICATION_HISTORY = []
MAX_ROUNDS = 3  # Checkpoint had 3 rounds
CURRENT_ROUND = 0

def save_step1_inputs(text):
    global TARGET_TEXT
    
    if not text.strip():
        return "❌ Error: Target Text cannot be empty."
        
    TARGET_TEXT = text
    
    return f'''✅ Saved Successfully!
    
    Target Text Length: {len(text)} chars
    
    You can now stop this cell and run Step 2.'''

with gr.Blocks() as step1_demo:
    gr.Markdown("## Step 1: Input Target Text")
    gr.Markdown("Please paste the LLM generated text you want to evaluate below.")
    
    txt_input = gr.Textbox(
        label="Target Text", 
        lines=10, 
        placeholder="Paste the LLM generated text here..."
    )
    
    submit_btn = gr.Button("Confirm & Save Text", variant="primary")
    output_msg = gr.Markdown()
    
    submit_btn.click(
        fn=save_step1_inputs, 
        inputs=[txt_input], 
        outputs=output_msg
    )

print("Launching Step 1 Interface...")
try:
    step1_demo.launch(height=500, inline=True, quiet=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


Launching Step 1 Interface...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 2: Qualification Assessment
The AI will now analyze the text you provided in Step 1 and assess your qualification to evaluate it.

In [11]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr
from src.utils import parse_json_output

def on_start_interview():
    global QUALIFICATION_HISTORY, CURRENT_ROUND, HUMAN_COMPETENCY_SCORE
    
    if not TARGET_TEXT:
        return [{"role": "assistant", "content": "❌ Error: No Target Text found. Please complete Step 1 first."}], gr.update(interactive=False)

    QUALIFICATION_HISTORY = []
    CURRENT_ROUND = 0
    
    try:
        try:
            prompt = QUALIFICATION_PROMPT.format(target_text=TARGET_TEXT, evaluation_purpose="General Assessment", history="")
        except KeyError:
            prompt = QUALIFICATION_PROMPT.format(target_text=TARGET_TEXT, history="")
            
        print("Calling LLM...")
        response = llm.generate("qwen-max", [{"role": "user", "content": prompt}])
        data = parse_json_output(response)
        
        status = data.get("status")
        if status == "asking":
            question = data.get("question")
            QUALIFICATION_HISTORY.append(f"AI: {question}")
            CURRENT_ROUND += 1
            # Messages format: List of Dicts
            return [{"role": "assistant", "content": f"🤖 Examiner: {question}"}], gr.update(interactive=True)
        else:
            score = data.get("score", 0)
            reason = data.get("reason", "Evaluation ended.")
            HUMAN_COMPETENCY_SCORE = score
            return [{"role": "assistant", "content": f"✅ Assessment Complete.\n**Score:** {score}\n**Reason:** {reason}"}], gr.update(interactive=False)
            
    except Exception as e:
        return [{"role": "assistant", "content": f"Error calling LLM: {str(e)}"}], gr.update(interactive=False)

def on_user_reply(user_input, history):
    global QUALIFICATION_HISTORY, CURRENT_ROUND, HUMAN_COMPETENCY_SCORE
    
    if not user_input.strip():
        return history, ""
    
    if history is None:
        history = []
        
    # Append User Message
    history.append({"role": "user", "content": user_input})
    QUALIFICATION_HISTORY.append(f"User: {user_input}")
    
    history_str = "\n".join(QUALIFICATION_HISTORY)
    try:
        try:
            prompt = QUALIFICATION_PROMPT.format(target_text=TARGET_TEXT, evaluation_purpose="General Assessment", history=history_str)
        except KeyError:
            prompt = QUALIFICATION_PROMPT.format(target_text=TARGET_TEXT, history=history_str)
            
        response = llm.generate("qwen-max", [{"role": "user", "content": prompt}])
        data = parse_json_output(response)
        
        status = data.get("status")
        
        if status == "asking" and CURRENT_ROUND < MAX_ROUNDS:
            question = data.get("question")
            QUALIFICATION_HISTORY.append(f"AI: {question}")
            CURRENT_ROUND += 1
            
            # Append Assistant Message
            history.append({"role": "assistant", "content": f"🤖 Examiner: {question}"})
            return history, ""
        else:
            score = data.get("score", 0)
            reason = data.get("reason", "Evaluation ended.")
            HUMAN_COMPETENCY_SCORE = score
            
            final_msg = f"✅ Assessment Complete.\n**Score:** {score}\n**Reason:** {reason}\n\nYou can now proceed to Step 3."
            history.append({"role": "assistant", "content": final_msg})
            return history, ""
            
    except Exception as e:
        history.append({"role": "assistant", "content": f"Error calling LLM: {str(e)}"})
        return history, ""

with gr.Blocks() as step2_demo:
    gr.Markdown("## Step 2: Qualification Assessment")
    gr.Markdown("Click **Start Assessment** to begin.")
    
    with gr.Row():
        start_btn = gr.Button("Start Assessment", variant="primary")
        clear_btn = gr.ClearButton(value="Reset")
    
    # Passing Dictionaries WITHOUT type="messages" to be safe.
    chatbot = gr.Chatbot(height=500, label="Interview Chat")
    msg = gr.Textbox(label="Your Answer", placeholder="Type here...", interactive=False)
    
    start_btn.click(on_start_interview, outputs=[chatbot, msg])
    msg.submit(on_user_reply, [msg, chatbot], [chatbot, msg])
    clear_btn.click(lambda: None, None, chatbot, queue=False)

print("Launching Step 2 Interface...")
try:
    step2_demo.launch(height=600, inline=True, quiet=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")


Launching Step 2 Interface...


INFO:httpx:HTTP Request: GET http://127.0.0.1:7862/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7862/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Step 3: Jury Evaluation
Now that your qualification is established, we proceed to the jury evaluation.

In [ ]:
# Widgets for Jury Step
human_score_input = widgets.FloatSlider(value=80, min=0, max=100, description='Your Score:')
human_reason_input = widgets.Textarea(description='Reason:', placeholder='Why did you give this score?')
start_jury_btn = widgets.Button(description='Start Jury Evaluation', button_style='success')
jury_output_area = widgets.Output()

display(human_score_input, human_reason_input, start_jury_btn, jury_output_area)

thread = {"configurable": {"thread_id": "1"}}

def on_start_jury(b):
    with jury_output_area:
        clear_output()
        if not TARGET_TEXT:
            print("Error: No Target Text found. Please complete Step 1.")
            return
            
        print(f"Initializing Jury with Human Competency Score: {HUMAN_COMPETENCY_SCORE}...")
        
        # Map Competency Score (0-100) to Weight (0.5-1.5)
        human_weight = 0.5 + (HUMAN_COMPETENCY_SCORE / 100.0)
        
        initial_state = {
            "topic": TARGET_TEXT,
            "human_bio": "Assessed via Qualification Exam",
            "human_score": human_score_input.value,
            "human_reason": human_reason_input.value,
            "human_weight": human_weight,
            "model_outputs": {},
            "debate_logs": [],
            "votes": {}
        }
        
        for event in app.stream(initial_state, thread, stream_mode="values"):
            if 'debate_logs' in event and event['debate_logs']:
                print("--- Debate Log ---")
                for log in event['debate_logs']:
                    print(log)
            if 'anonymized_reasons' in event:
                print("Ready for Voting...")
                
        # Check state after interruption
        state_snapshot = app.get_state(thread)
        if state_snapshot.next:
            print("System paused for Human Vote.")
            show_voting_ui(state_snapshot.values)

start_jury_btn.on_click(on_start_jury)

## Step 4: Voting Interface

In [ ]:
vote_widget = widgets.RadioButtons(options=[], description='Best Reason:')
submit_vote_btn = widgets.Button(description='Submit Vote', button_style='info')
vote_output = widgets.Output()

def show_voting_ui(state_values):
    options = state_values['anonymized_reasons']
    display_options = []
    for opt_id, text in options.items():
        display_options.append((f"{opt_id}: {text[:100]}...", opt_id))
    
    vote_widget.options = display_options
    display(vote_widget, submit_vote_btn, vote_output)

def on_vote_submit(b):
    with vote_output:
        selected_option = vote_widget.value
        print(f"You voted for: {selected_option}")
        
        current_votes = app.get_state(thread).values.get('votes', {})
        current_votes['human'] = selected_option
        
        app.update_state(thread, {"votes": current_votes})
        
        print("Resuming execution...")
        for event in app.stream(None, thread, stream_mode="values"):
            if 'final_verdict' in event:
                display(Markdown("### Final Verdict"))
                display(Markdown(event['final_verdict']))

submit_vote_btn.on_click(on_vote_submit)